# Assignment 3 - Anomaly detection

This notebook compares anomaly-detection approaches and writes `anomalies.csv`.
The key design decision is to use structural features, not only topical TF-IDF,
because the suspicious files are described as unsafe/corrupted files.


## Run 1 - Analysis / rationale

Load data and build features. The feature family targets unsafe marketplace
structure: listing IDs, URLs, e-commerce tokens, repeated pickup/discount
phrases, suspicious domains, and simple formatting ratios.


In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "articles.csv").exists() and (ROOT.parent / "articles.csv").exists():
    ROOT = ROOT.parent
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)

def clean_text_for_clustering(text: str) -> str:
    """Remove artifacts that are not topical content.

    The goal is not to erase the document meaning. The goal is to stop the model
    from clustering on Usenet signatures, URLs, e-mail addresses, and random IDs.
    """
    text = str(text)

    # Common Usenet signature separator. Apply only if the separator is late
    # enough to plausibly be a signature, not a normal sentence break.
    matches = list(re.finditer(r"\s--\s", text))
    if matches:
        last = matches[-1]
        if last.start() > len(text) * 0.45 or len(text) - last.start() < 900:
            text = text[: last.start()]

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", " ", text)
    text = re.sub(r"\b[A-Z0-9]{5,}(?:-[A-Z0-9]{2,})+\b", " ", text)
    text = re.sub(r"[_=]{2,}|-{3,}", " ", text)
    return text


def build_unsafe_features(texts: pd.Series) -> pd.DataFrame:
    """Features aimed at unsafe/corrupted marketplace-like documents.

    These are intentionally different from topic-clustering features. A document
    can be anomalous because of structure and repeated machine-like tokens, even
    if its individual words are common.
    """
    rows = []
    for text in texts.astype(str):
        urls = len(re.findall(r"https?://\S+|www\.\S+", text))
        listing = len(re.findall(r"LISTING_ID_", text))
        ecommerce = len(
            re.findall(
                r"\b(?:SKU|TOKEN|COUPON|PROMO(?:_ID|_REF)?|BUNDLE_REF|ORDER_REF|SELLER_CODE|PRICE_REF|SALE_CODE)\b",
                text,
            )
        )
        contact_market = len(
            re.findall(
                r"Contact:|\+32\s?\d|asking\s+\d+|euro|EUR|pickup|discount|coupon|offer|sale|cash only|quick pickup|fast deal|special discount",
                text,
                flags=re.I,
            )
        )
        bad_domains = len(
            re.findall(
                r"pickup-offer-fast|trusted-sales-direct|bonus-marketplace|clearance-deals|claim-discount-now|deal-bundle-now",
                text,
                flags=re.I,
            )
        )
        html_ui = len(
            re.findall(
                r"<(?:div|span|button|input|a)\b|</(?:div|span|button|a)>",
                text,
                flags=re.I,
            )
        )
        chars = list(text)
        char_len = len(text)
        digit_ratio = sum(c.isdigit() for c in chars) / max(char_len, 1)
        upper_ratio = sum(c.isupper() for c in chars) / max(char_len, 1)
        punct_ratio = sum((not c.isalnum() and not c.isspace()) for c in chars) / max(char_len, 1)
        words = re.findall(r"[A-Za-z]+", text)
        unique_word_ratio = len(set(w.lower() for w in words)) / max(len(words), 1)

        unsafe_score = (
            10 * listing
            + 3 * urls
            + 2 * ecommerce
            + contact_market
            + 4 * bad_domains
            + html_ui
        )
        rows.append(
            {
                "char_len": char_len,
                "word_count": len(words),
                "unique_word_ratio": unique_word_ratio,
                "digit_ratio": digit_ratio,
                "upper_ratio": upper_ratio,
                "punct_ratio": punct_ratio,
                "url_count": urls,
                "listing_marker_count": listing,
                "ecommerce_token_count": ecommerce,
                "market_phrase_count": contact_market,
                "bad_domain_count": bad_domains,
                "html_ui_token_count": html_ui,
                "unsafe_score": unsafe_score,
            }
        )
    return pd.DataFrame(rows)

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.preprocessing import RobustScaler

articles = pd.read_csv(ROOT / "articles.csv")
anomalies_template = pd.read_csv(ROOT / "anomalies.csv")
assert anomalies_template.shape[0] == 50

features = build_unsafe_features(articles["text"])
features.insert(0, "doc_id", articles["doc_id"])
features.to_csv(OUTPUT_DIR / "anomaly_structural_features.csv", index=False)

display(
    features[
        [
            "url_count",
            "listing_marker_count",
            "ecommerce_token_count",
            "market_phrase_count",
            "bad_domain_count",
            "html_ui_token_count",
            "unsafe_score",
        ]
    ].quantile([0, 0.5, 0.95, 0.98, 0.99, 1.0])
)


## Run 2 - Analysis / rationale

Why not detect anomalies only with TF-IDF? A normal code snippet or hockey table
can be textually unusual, but it is not necessarily an unsafe download. The
assignment explicitly mentions structure, formatting, repetition, and corrupted
patterns. Therefore the model should score those properties directly.

I compare three unsupervised signals:

- Isolation Forest: isolates rare structural combinations.
- k-nearest-neighbor distance: flags points far from their neighbors.
- LOF: checks local density. LOF is included, but it can be less reliable here
  because legitimate subgenres such as source code or sports tables can form
  local sparse regions.


In [ ]:
feature_cols = [
    "url_count",
    "listing_marker_count",
    "ecommerce_token_count",
    "market_phrase_count",
    "bad_domain_count",
    "html_ui_token_count",
    "unsafe_score",
]

X = RobustScaler().fit_transform(np.log1p(features[feature_cols]))
contamination = 50 / len(features)

iso = IsolationForest(n_estimators=500, contamination=contamination, random_state=42)
iso.fit(X)
features["isolation_score"] = -iso.decision_function(X)
iso_idx = set(features["isolation_score"].nlargest(50).index)

knn = NearestNeighbors(n_neighbors=20, metric="euclidean")
knn.fit(X)
knn_distances, _ = knn.kneighbors(X)
features["knn_20_distance"] = knn_distances[:, -1]
knn_idx = set(features["knn_20_distance"].nlargest(50).index)

lof = LocalOutlierFactor(n_neighbors=20, contamination=contamination)
lof.fit_predict(X)
features["lof_score"] = -lof.negative_outlier_factor_
lof_idx = set(features["lof_score"].nlargest(50).index)

score_idx = set(features["unsafe_score"].nlargest(50).index)

comparison = pd.DataFrame(
    [
        {
            "method": "IsolationForest",
            "selected": len(iso_idx),
            "overlap_with_structural_score_top50": len(iso_idx & score_idx),
            "overlap_with_knn_top50": len(iso_idx & knn_idx),
        },
        {
            "method": "kNN distance",
            "selected": len(knn_idx),
            "overlap_with_structural_score_top50": len(knn_idx & score_idx),
            "overlap_with_knn_top50": len(knn_idx & knn_idx),
        },
        {
            "method": "LOF",
            "selected": len(lof_idx),
            "overlap_with_structural_score_top50": len(lof_idx & score_idx),
            "overlap_with_knn_top50": len(lof_idx & knn_idx),
        },
    ]
)
comparison.to_csv(OUTPUT_DIR / "anomaly_detection_comparison.csv", index=False)
display(comparison)


## Run 3 - Analysis / rationale

Skeptical ablation: the full model includes `listing_marker_count`, because the
data itself contains that machine-like marker. To ensure the result is not just
a brittle exact-regex shortcut, I rerun Isolation Forest and kNN without
`listing_marker_count` and without the combined `unsafe_score`. If the same
documents are still selected, the evidence comes from independent unsafe
structure such as URLs, e-commerce tokens, suspicious domains, and repeated
marketplace phrases.


In [ ]:
ablation_feature_sets = {
    "full_structural": feature_cols,
    "without_listing_marker_and_unsafe_score": [
        "url_count",
        "ecommerce_token_count",
        "market_phrase_count",
        "bad_domain_count",
        "html_ui_token_count",
    ],
}

ablation_rows = []
for name, cols in ablation_feature_sets.items():
    X_ablation = RobustScaler().fit_transform(np.log1p(features[cols]))

    iso_ablation = IsolationForest(n_estimators=500, contamination=contamination, random_state=42)
    iso_ablation.fit(X_ablation)
    iso_ablation_idx = set(np.argsort(iso_ablation.decision_function(X_ablation))[:50])

    knn_ablation = NearestNeighbors(n_neighbors=20, metric="euclidean")
    knn_ablation.fit(X_ablation)
    ablation_distances, _ = knn_ablation.kneighbors(X_ablation)
    knn_ablation_idx = set(np.argsort(ablation_distances[:, -1])[-50:])

    ablation_rows.append(
        {
            "feature_set": name,
            "iforest_overlap_with_final": len(iso_ablation_idx & iso_idx),
            "knn_overlap_with_final": len(knn_ablation_idx & iso_idx),
            "iforest_selected": len(iso_ablation_idx),
            "knn_selected": len(knn_ablation_idx),
        }
    )

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(OUTPUT_DIR / "anomaly_ablation_check.csv", index=False)
display(ablation_df)


## Run 4 - Analysis / rationale

The final method is Isolation Forest on unsafe-structure features. It agrees
with the direct structural-score ranking and kNN distance on the 50 strongest
unsafe-format documents. That agreement is stronger evidence than any single
method alone.


In [ ]:
final_idx = sorted(iso_idx)
final_anomalies = articles.iloc[final_idx][["doc_id"]].copy()
final_anomalies.insert(0, "anomaly", range(1, len(final_anomalies) + 1))

ranked_details = features.iloc[final_idx].copy()
ranked_details = ranked_details.sort_values("isolation_score", ascending=False)
ranked_details.to_csv(OUTPUT_DIR / "anomaly_candidates_ranked.csv", index=False)

display(ranked_details[["doc_id", *feature_cols, "isolation_score", "knn_20_distance", "lof_score"]].head(15))


## Run 5 - Analysis / rationale

Write and validate `anomalies.csv`. The submission must contain exactly 50
document IDs, all present in `articles.csv`, without duplicates or nulls.


In [ ]:
assert final_anomalies.shape == (50, 2)
assert final_anomalies["doc_id"].is_unique
assert final_anomalies["doc_id"].isin(articles["doc_id"]).all()
assert final_anomalies["doc_id"].notna().all()

final_anomalies.to_csv(ROOT / "anomalies.csv", index=False)

written = pd.read_csv(ROOT / "anomalies.csv")
assert written.shape == (50, 2)
assert written["doc_id"].is_unique
assert written["doc_id"].isin(articles["doc_id"]).all()

print("Wrote:", ROOT / "anomalies.csv")
display(written.head(20))
